# Customer Intelligence & Support Copilot
## Instructor-led end-to-end notebook

**Scenario:** synthetic grocery quick-commerce data. This notebook emphasizes *how data is fetched, inspected, cleaned, validated, stored, and then consumed by downstream AI/ML components*.

### Before class
1. Run `python generate_data.py` from the repository root.
2. Install packages with `pip install -r requirements.txt`.
3. Run this notebook from the repository root so relative paths work.

All data and policies are fictional. The LLM calls are handled by `mock_llm.py` locally; no API key is required.

## 0. Repository map and end-to-end flow

`generate_data.py` → `data/raw/*.csv` → audit → clean → `data/clean/*.csv` + `data/instamart.db` → RFM/segments + lapse model → privacy-safe ticket triage → policy retrieval + refund tools.

Teaching point: **fetching data** does not always mean calling an API. Here, the source is a local CSV export. In a production system, the same ingestion layer could use an approved API or warehouse connector, but credentials and network access are intentionally out of scope.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "generate_data.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)
print("Root files:", sorted(p.name for p in ROOT.iterdir() if p.is_file()))

Project root: c:\instamart_customer_copilot_teaching_kit
Root files: ['.gitignore', 'GITHUB_RESOURCES.md', 'README.md', 'generate_data.py', 'mock_llm.py', 'requirements.txt', 'run_project.py']


## 1. Generate / fetch the raw data

The data generator is deterministic (fixed random seed). It writes customer, product, order, support-ticket, evaluation-question, agent-scenario inputs and policy text files.

Run it once when starting from a clean clone. Running it again recreates the same synthetic source exports.

In [2]:
import subprocess, sys
subprocess.run([sys.executable, str(ROOT / "generate_data.py")], check=True)
raw_dir = ROOT / "data" / "raw"
print("Raw files:", sorted(p.name for p in raw_dir.iterdir()))

Raw files: ['customers.csv', 'orders.csv', 'policy_eval.csv', 'products.csv', 'tickets.csv']


## 2. Read the exports and understand their shape

Use `keep_default_na=False` for ticket text so empty strings remain empty strings. This is important because empty tickets must be identified by an explicit guardrail rather than silently changing the input representation.

In [3]:
import pandas as pd
customers_raw = pd.read_csv(raw_dir / "customers.csv", keep_default_na=False)
products_raw = pd.read_csv(raw_dir / "products.csv", keep_default_na=False)
orders_raw = pd.read_csv(raw_dir / "orders.csv", keep_default_na=False)
tickets_raw = pd.read_csv(raw_dir / "tickets.csv", keep_default_na=False)

for name, df in [("customers", customers_raw), ("products", products_raw),
                 ("orders", orders_raw), ("tickets", tickets_raw)]:
    print(f"\\n{name}: shape={df.shape}")
    display(df.head(3))
    display(pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "blank_count": df.astype(str).apply(lambda col: col.str.strip().eq("").sum()),
        "missing_pct": (df.isna().mean() * 100).round(2)
    }))

\ncustomers: shape=(1200, 5)


,customer_id,city,signup_date,age,gender
0,C0001,Chennai,2024-01-13,35,MALE
1,C0002,DELHI,2024-02-17,31,MALE
2,C0003,bengaluru,2025-02-18,18,F


,dtype,missing_count,blank_count,missing_pct
customer_id,str,0,0,0.0
city,str,0,0,0.0
signup_date,str,0,0,0.0
age,str,0,61,0.0
gender,str,0,30,0.0


\nproducts: shape=(40, 4)


,product_id,product_name,category,price_inr
0,P001,Basmati Rice 1kg,Grocery,129
1,P002,Toor Dal 1kg,Grocery,165
2,P003,Atta 5kg,Grocery,245


,dtype,missing_count,blank_count,missing_pct
product_id,str,0,0,0.0
product_name,str,0,0,0.0
category,str,0,0,0.0
price_inr,int64,0,0,0.0


... (2 outputs left out)

,dtype,missing_count,blank_count,missing_pct
order_id,str,0,0,0.0
customer_id,str,0,0,0.0
product_id,str,0,0,0.0
quantity,int64,0,0,0.0
order_ts,str,0,0,0.0
status,str,0,0,0.0
discount_pct,int64,0,0,0.0
order_amount,str,0,0,0.0
delivery_minutes,str,0,2541,0.0


\ntickets: shape=(164, 5)


,ticket_id,customer_id,order_id,text,true_category
0,T0024,C0486,ORD-105035,There is an unauthorized transaction on my car...,fraud
1,T0073,C0080,ORD-100816,The invoice for ORD-100816 shows a wrong amoun...,billing
2,T0095,C1087,ORD-111156,The payment page shows an error and the app fr...,technical


,dtype,missing_count,blank_count,missing_pct
ticket_id,str,0,0,0.0
customer_id,str,0,0,0.0
order_id,str,0,8,0.0
text,str,0,4,0.0
true_category,str,0,0,0.0


### Audit questions for students
- Which columns are supposed to be unique keys?
- Are duplicates exact duplicates or multiple states of the same order?
- Which values are impossible (negative delivery duration, non-numeric money, implausible age)?
- Why should we distinguish an empty string from a missing value?
- What would go wrong if we modelled before cleaning?

In [4]:
from src.data_pipeline import audit, clean_data, validate
audit()


customers: shape=(1200, 5)
customer_id    str
city           str
signup_date    str
age            str
gender         str
dtype: object
             missing_count  missing_pct
customer_id              0          0.0
city                     0          0.0
signup_date              0          0.0
age                      0          0.0
gender                   0          0.0

products: shape=(40, 4)
product_id        str
product_name      str
category          str
price_inr       int64
dtype: object
              missing_count  missing_pct
product_id                0          0.0
product_name              0          0.0
category                  0          0.0
price_inr                 0          0.0

orders: shape=(13084, 9)
order_id              str
customer_id           str
product_id            str
quantity            int64
order_ts              str
status                str
discount_pct        int64
order_amount          str
delivery_minutes      str
dtype: object
                 

## 3. Clean the data and write trusted outputs

The pipeline applies explicit rules:
- standardize city and gender;
- make impossible ages missing and fill with city median;
- parse timestamps, remove exact duplicates, then keep the latest row per order ID;
- coerce invalid amounts to missing and calculate replacements from product price × quantity × discount;
- set negative delivery times to missing and impute only completed orders using product-category medians;
- save clean CSVs and SQLite tables.

Ask students to explain why cancelled orders should keep a missing delivery duration.

In [5]:
clean = clean_data()
for name, df in clean.items():
    print(name, df.shape)
    display(df.head(3))
validate()

customers (1200, 5)


,customer_id,city,signup_date,age,gender
0,C0001,Chennai,2024-01-13,35,male
1,C0002,Delhi,2024-02-17,31,male
2,C0003,Bengaluru,2025-02-18,18,female


products (40, 4)


,product_id,product_name,category,price_inr
0,P001,Basmati Rice 1kg,Grocery,129
1,P002,Toor Dal 1kg,Grocery,165
2,P003,Atta 5kg,Grocery,245


orders (12309, 9)


,order_id,customer_id,product_id,quantity,order_ts,status,discount_pct,order_amount,delivery_minutes
827,ORD-100001,C0001,P001,1,2025-09-16 08:07:21,completed,15,109.65,20
4294,ORD-100002,C0001,P008,1,2025-09-03 07:34:01,completed,40,25.20,23
9973,ORD-100003,C0001,P008,1,2025-04-08 17:08:14,completed,0,42.00,23


ALL CHECKPOINTS PASSED


## 4. Query the SQLite database

The clean tables are persisted in `data/instamart.db`. SQLite gives us a portable analytical database that can be queried without setting up a server.

Run the supplied SQL examples and compare SQL totals to pandas results.

In [6]:

import sqlite3
import pandas as pd

db_path = ROOT / "data" / "instamart.db"

with sqlite3.connect(db_path) as con:
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
        con
    )
    display(tables)

    q2 = pd.read_sql_query("""
        SELECT c.city, p.category,
               ROUND(SUM(o.order_amount), 2) AS revenue
        FROM orders o
        JOIN customers c USING(customer_id)
        JOIN products p USING(product_id)
        WHERE o.status = 'completed'
        GROUP BY c.city, p.category
        ORDER BY c.city, p.category
    """, con)

display(q2)
print("SQL grand total INR:", round(q2["revenue"].sum(), 2))


,name
0,customers
1,orders
2,products


,city,category,revenue
0,Bengaluru,Dairy,87075.05
1,Bengaluru,Electronics,233909.45
2,Bengaluru,Grocery,110915.65
3,Bengaluru,Personal Care,20193.85
4,Bengaluru,Snacks,71549.40
5,Chennai,Dairy,78301.80
6,Chennai,Electronics,195508.80
7,Chennai,Grocery,93817.45
8,Chennai,Personal Care,27675.40
9,Chennai,Snacks,68325.70


SQL grand total INR: 2998668.8


## 5. Load and join the clean data



In [7]:
import pandas as pd
import numpy as np

In [8]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

ROOT = Path.cwd()

if not (ROOT / "generate_data.py").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)

Project root: c:\instamart_customer_copilot_teaching_kit


In [9]:

customers = pd.read_csv(ROOT / "data/clean/customers.csv")
products = pd.read_csv(ROOT / "data/clean/products.csv")
orders = pd.read_csv(ROOT / "data/clean/orders.csv")

completed = orders[orders["status"] == "completed"].copy()

df = (
    completed
    .merge(customers[["customer_id", "city"]], on="customer_id")
    .merge(products[["product_id", "category"]], on="product_id")
)

print("Completed orders:", len(df))


Completed orders: 11110


## Pandas pivot table


In [10]:

pivot = pd.pivot_table(
    df,
    index="city",
    columns="category",
    values="order_amount",
    aggfunc="sum",
    fill_value=0,
    margins=True
)

display(pivot)
print("Pandas grand total:", round(pivot.loc["All", "All"], 2))
print("SQL grand total:", round(q2["revenue"].sum(), 2))


category,Dairy,Electronics,Grocery,Personal Care,Snacks,All
city,,,,,,
Bengaluru,87075.05,233909.45,110915.65,20193.85,71549.40,523643.40
Chennai,78301.80,195508.80,93817.45,27675.40,68325.70,463629.15
Delhi,117271.70,158549.10,112658.05,23471.50,90879.45,502829.80
Hyderabad,83731.00,228913.10,109475.40,24800.05,81670.35,528589.90
Mumbai,75209.85,214738.40,78970.15,38018.00,72122.10,479058.50
Pune,90349.55,187629.25,108456.60,34275.15,80207.50,500918.05
All,531938.95,1219248.10,614293.30,168433.95,464754.50,2998668.80


Pandas grand total: 2998668.8
SQL grand total: 2998668.8


## City-level aggregation

In [11]:

city_stats = df.groupby("city").agg(
    total_revenue=("order_amount", "sum"),
    average_order=("order_amount", "mean"),
    order_count=("order_amount", "count")
)

display(city_stats.round(2))


,total_revenue,average_order,order_count
city,,,
Bengaluru,523643.40,276.47,1894
Chennai,463629.15,272.72,1700
Delhi,502829.80,243.62,2064
Hyderabad,528589.90,284.95,1855
Mumbai,479058.50,297.00,1613
Pune,500918.05,252.48,1984


## Outlier analysis


In [12]:

amount = df["order_amount"]

q1 = amount.quantile(0.25)
q3 = amount.quantile(0.75)
iqr = q3 - q1

lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

print("Mean:", round(amount.mean(), 2))
print("Median:", round(amount.median(), 2))
print("Skewness:", round(amount.skew(), 2))
print("Q1:", round(q1, 2))
print("Q3:", round(q3, 2))
print("Lower fence:", round(lower_fence, 2))
print("Upper fence:", round(upper_fence, 2))
print("Upper outliers:", (amount > upper_fence).sum())
print("Lower outliers:", (amount < lower_fence).sum())


Mean: 269.91
Median: 135.0
Skewness: 4.61
Q1: 64.8
Q3: 297.0
Lower fence: -283.5
Upper fence: 645.3
Upper outliers: 897
Lower outliers: 0


## Correlation matrix

In [13]:

correlation = df[
    ["order_amount", "discount_pct", "quantity", "delivery_minutes"]
].corr()

display(correlation.round(3))


,order_amount,discount_pct,quantity,delivery_minutes
order_amount,1.000,-0.095,0.087,-0.003
discount_pct,-0.095,1.000,-0.000,-0.010
quantity,0.087,-0.000,1.000,-0.013
delivery_minutes,-0.003,-0.010,-0.013,1.000



## Part A — Exploratory Data Analysis (EDA)

### 1. What is this data about?
The dataset contains synthetic grocery orders across six cities, including order amounts, product categories, discounts, quantities and delivery times.

### 2. What do its distributions and statistics look like?
The mean completed order amount is INR 269.91, while the median is INR 135.00. The skewness is approximately 4.61, indicating a right-skewed distribution.

### 3. Are there outliers or anomalies?
The first quartile is INR 64.80 and the third quartile is INR 297.00. The upper IQR fence is INR 645.30. There are 897 upper outliers and no lower outliers.

### 4. What relationships exist between variables?
Order amount and discount percentage have the strongest absolute correlation, approximately -0.095. This is a weak negative relationship. Correlation does not establish causation.


## 6. Part B — customer segments and lapse prediction

RFM means:
- **Recency:** days since the most recent completed order at the snapshot date.
- **Frequency:** number of distinct completed orders.
- **Monetary:** total spend across completed orders.

The model module scales RFM for K-means, compares K values, uses PCA to inspect compressed variance, and builds a lapse model. The lapse model's input features use only orders before 2025-09-01; September is reserved to define the label. The scaler and encoder are fitted inside a training pipeline, not on the test data.

In [14]:
import sys
print(sys.executable)

import joblib
print("Joblib:", joblib.__version__)

c:\instamart_customer_copilot_teaching_kit\.venv\Scripts\python.exe
Joblib: 1.6.0


In [15]:
from src.customer_models import rfm_and_segments, build_lapse_model
rfm, k_diagnostics, segment_profile, pca_variance = rfm_and_segments(k=4)
display(k_diagnostics.round(3))
display(segment_profile.round(2))
print("PCA explained variance ratio:", pca_variance, "total:", pca_variance.sum())
display(rfm.head())
lapse_summary = build_lapse_model()
lapse_summary

,k,inertia,silhouette
0,2,1820.761,0.465
1,3,1268.235,0.445
2,4,743.071,0.526
3,5,617.367,0.480
4,6,503.171,0.482
5,7,408.460,0.416
6,8,366.503,0.412


,segment_id,size,recency,frequency,monetary
0,0,436,28.86,3.77,1057.62
1,1,138,24.12,6.77,7129.70
2,2,419,8.77,19.40,3410.02
3,3,182,109.04,2.20,685.99


PCA explained variance ratio: [0.63427875 0.21803762] total: 0.8523163667502166


,customer_id,frequency,monetary,recency,segment_id,segment_name,is_premium
0,C0001,4,512.85,14,0,At Risk,False
1,C0002,7,4644.90,28,1,Premium,True
2,C0003,22,3644.85,7,2,Frequent,False
3,C0004,5,369.00,0,0,At Risk,False
4,C0005,2,324.00,147,3,At Risk,False



Frequency mean learned by training scaler: 8.114718614718615
Full cohort frequency mean: 8.028546712802768

Frequency mean learned by training scaler: 8.114718614718615
Full cohort frequency mean: 8.028546712802768
Original training class counts:
lapsed
0    570
1    354
Name: count, dtype: int64

Balanced training class counts:
lapsed
1    570
0    570
Name: count, dtype: int64

Original Logistic Regression F1: 0.6480446927374302
Oversampled Logistic Regression:
Precision: 0.5891472868217055
Recall: 0.8539325842696629
F1: 0.6972477064220184
Confusion Matrix: [[90, 53], [13, 76]]

Top 5 Random Forest Feature Importances:
num__monetary             0.280428
num__frequency            0.277911
num__recency              0.109118
num__avg_delivery         0.059732
num__cancellation_rate    0.051834
dtype: float64

Top 3 Logistic Regression Coefficients:
num__frequency                        -1.697333
cat__favourite_category_Electronics   -1.135167
cat__favourite_category_Grocery       -0.44

{'cohort_rows': 1156,
 'lapsed': 443,
 'baseline_accuracy': 0.6163793103448276,
 'models': {'logistic': {'best_params': {'model__C': 100},
   'cv_f1': np.float64(0.650552962900789),
   'precision': 0.6444444444444445,
   'recall': 0.651685393258427,
   'f1': 0.6480446927374302,
   'confusion_matrix': [[111, 32], [31, 58]]},
  'forest': {'best_params': {'model__max_depth': 8,
    'model__min_samples_leaf': 10},
   'cv_f1': np.float64(0.6865517479789224),
   'precision': 0.6333333333333333,
   'recall': 0.6404494382022472,
   'f1': 0.6368715083798883,
   'confusion_matrix': [[110, 33], [32, 57]]}},
 'retention_list':     customer_id  lapse_probability  actually_lapsed
 13        C0862           0.510513                0
 190       C0267           0.477212                0
 165       C1008           0.473260                0
 227       C0060           0.471746                0
 5         C0802           0.471582                0
 216       C0942           0.465245                0
 144   

In [16]:
print("RFM shape:", rfm.shape)

print(rfm[["recency", "frequency", "monetary"]].describe())

RFM shape: (1175, 7)
           recency    frequency      monetary
count  1175.000000  1175.000000   1175.000000
mean     33.558298     9.455319   2552.058553
std      38.433705     8.009971   2367.305104
min       0.000000     1.000000     12.000000
25%       6.000000     3.000000    505.450000
50%      19.000000     5.000000   2286.500000
75%      46.000000    17.000000   3751.525000
max     180.000000    33.000000  14766.550000


### Instructor discussion: leakage
Ask students to identify:
1. Which dates can enter the lapse-model features?
2. Why must September orders not enter the features?
3. Why is a Premium-only retention list labelled *illustrative* if the Premium flag comes from whole-window RFM?
4. Why can accuracy be misleading when a substantial minority of customers lapse?

## 6. Part C — support ticket triage

This pipeline demonstrates:
1. input guardrails for empty and overlong tickets;
2. ordered PII redaction (email → phone → order ID);
3. retry logic for simulated transient errors;
4. strict category validation and a keyword fallback;
5. token budgeting and human-review routing.

The exact user message sent to the mock classifier is the redacted ticket text only.

In [17]:
from src.triage import run_pipeline, FEW, ZERO
triage_results, triage_summary = run_pipeline(prompt=FEW, token_budget=100000)
print(triage_summary)
display(triage_results.head(10))
display(triage_results.status.value_counts().rename_axis("status").reset_index(name="tickets"))

retry ticket=T0002 retry=0 wait=0.0530s
retry ticket=T0116 retry=0 wait=0.0453s
retry ticket=T0153 retry=0 wait=0.0484s
retry ticket=T0039 retry=0 wait=0.0478s
retry ticket=T0152 retry=0 wait=0.0542s
retry ticket=T0043 retry=0 wait=0.0535s
retry ticket=T0111 retry=0 wait=0.0562s
retry ticket=T0155 retry=0 wait=0.0461s
retry ticket=T0100 retry=0 wait=0.0503s
retry ticket=T0068 retry=0 wait=0.0454s
retry ticket=T0135 retry=0 wait=0.0477s
retry ticket=T0009 retry=0 wait=0.0513s
retry ticket=T0067 retry=0 wait=0.0453s
retry ticket=T0106 retry=0 wait=0.0475s
retry ticket=T0097 retry=0 wait=0.0531s
retry ticket=T0003 retry=0 wait=0.0518s
retry ticket=T0019 retry=0 wait=0.0478s
retry ticket=T0086 retry=0 wait=0.0524s
retry ticket=T0065 retry=0 wait=0.0551s
{'tokens': 12520, 'redactions': {'email': 55, 'phone': 91, 'order_id': 132}, 'status_counts': {'classified_json': 135, 'classified_keyword': 12, 'flagged_invalid_output': 9, 'skipped_empty': 4, 'flagged_long': 4}}


,ticket_id,customer_id,status,category,segment_name,priority,route,tokens_used
0,T0024,C0486,classified_json,fraud,At Risk,HIGH,security_team,84
1,T0073,C0080,classified_json,billing,Frequent,NORMAL,support_queue,80
2,T0095,C1087,classified_json,technical,At Risk,NORMAL,support_queue,82
3,T0159,C1172,skipped_empty,,Frequent,NORMAL,human_review,0
4,T0080,C0910,classified_json,fraud,At Risk,HIGH,security_team,84
5,T0124,C0139,classified_json,fraud,Frequent,HIGH,security_team,84
6,T0061,C0812,classified_json,billing,Frequent,NORMAL,support_queue,76
7,T0118,C0355,classified_json,delivery,Frequent,NORMAL,support_queue,79
8,T0037,C1156,classified_json,billing,Premium,HIGH,premium_desk,80
9,T0002,C0881,classified_json,delivery,At Risk,NORMAL,support_queue,78


,status,tickets
0,classified_json,135
1,classified_keyword,12
2,flagged_invalid_output,9
3,skipped_empty,4
4,flagged_long,4


### Discussion: privacy and safety
- Redaction is a defence-in-depth step, not a guarantee that every sensitive detail is removed.
- Why are invalid categories sent to human review instead of guessed?
- Why should fraud override ordinary customer-segment routing?
- Why do empty/long tickets need to be blocked before any model call?

In [21]:
from src.triage import run_pipeline, ZERO

result, stats = run_pipeline(prompt=ZERO, token_budget=100000)

print("Total tickets:", len(result))
print("\nStatus counts:")
print(result["status"].value_counts())

print("\nRedaction counts:")
print(stats["redactions"])

print("\nTotal tokens:", stats["tokens"])

retry ticket=T0002 retry=0 wait=0.0530s
retry ticket=T0116 retry=0 wait=0.0453s
retry ticket=T0153 retry=0 wait=0.0484s
retry ticket=T0039 retry=0 wait=0.0478s
retry ticket=T0152 retry=0 wait=0.0542s
retry ticket=T0043 retry=0 wait=0.0535s
retry ticket=T0111 retry=0 wait=0.0562s
retry ticket=T0155 retry=0 wait=0.0461s
retry ticket=T0100 retry=0 wait=0.0503s
retry ticket=T0068 retry=0 wait=0.0454s
retry ticket=T0135 retry=0 wait=0.0477s
retry ticket=T0009 retry=0 wait=0.0513s
retry ticket=T0067 retry=0 wait=0.0453s
retry ticket=T0106 retry=0 wait=0.0475s
retry ticket=T0097 retry=0 wait=0.0531s
retry ticket=T0003 retry=0 wait=0.0518s
retry ticket=T0019 retry=0 wait=0.0478s
retry ticket=T0086 retry=0 wait=0.0524s
retry ticket=T0065 retry=0 wait=0.0551s
Total tickets: 164

Status counts:
status
classified_json           135
classified_keyword         12
flagged_invalid_output      9
skipped_empty               4
flagged_long                4
Name: count, dtype: int64

Redaction counts:
{'e

In [22]:
from src.triage import run_pipeline, FEW

few_result, few_stats = run_pipeline(
    prompt=FEW,
    token_budget=100000
)

print("Few-shot status counts:")
print(few_result["status"].value_counts())

print("\nFew-shot total tokens:", few_stats["tokens"])

retry ticket=T0002 retry=0 wait=0.0530s
retry ticket=T0116 retry=0 wait=0.0453s
retry ticket=T0153 retry=0 wait=0.0484s
retry ticket=T0039 retry=0 wait=0.0478s
retry ticket=T0152 retry=0 wait=0.0542s
retry ticket=T0043 retry=0 wait=0.0535s
retry ticket=T0111 retry=0 wait=0.0562s
retry ticket=T0155 retry=0 wait=0.0461s
retry ticket=T0100 retry=0 wait=0.0503s
retry ticket=T0068 retry=0 wait=0.0454s
retry ticket=T0135 retry=0 wait=0.0477s
retry ticket=T0009 retry=0 wait=0.0513s
retry ticket=T0067 retry=0 wait=0.0453s
retry ticket=T0106 retry=0 wait=0.0475s
retry ticket=T0097 retry=0 wait=0.0531s
retry ticket=T0003 retry=0 wait=0.0518s
retry ticket=T0019 retry=0 wait=0.0478s
retry ticket=T0086 retry=0 wait=0.0524s
retry ticket=T0065 retry=0 wait=0.0551s
Few-shot status counts:
status
classified_json           135
classified_keyword         12
flagged_invalid_output      9
skipped_empty               4
flagged_long                4
Name: count, dtype: int64

Few-shot total tokens: 12520


In [23]:
import pandas as pd

tickets = pd.read_csv("data/raw/tickets.csv", keep_default_na=False)

def evaluate(result, name, total_tokens):
    comparison = result.merge(
        tickets[["ticket_id", "true_category"]],
        on="ticket_id",
        how="left"
    )

    classified = comparison[
        comparison["status"].isin(
            ["classified_json", "classified_keyword"]
        )
    ]

    accuracy = (
        classified["category"] == classified["true_category"]
    ).mean() * 100

    print(f"\n{name}")
    print("Classified tickets:", len(classified))
    print(f"Accuracy: {accuracy:.1f}%")
    print("Total tokens:", total_tokens)
    print(
        "Tokens per classified ticket:",
        round(total_tokens / len(classified), 2)
    )

    return comparison, classified

zero_comparison, zero_classified = evaluate(
    result, "ZERO-SHOT", stats["tokens"]
)

few_comparison, few_classified = evaluate(
    few_result, "FEW-SHOT", few_stats["tokens"]
)

print("\nZERO-SHOT CONFUSION TABLE")
print(pd.crosstab(
    zero_classified["true_category"],
    zero_classified["category"],
    rownames=["Actual"],
    colnames=["Predicted"]
))


ZERO-SHOT
Classified tickets: 147
Accuracy: 85.7%
Total tokens: 7372
Tokens per classified ticket: 50.15

FEW-SHOT
Classified tickets: 147
Accuracy: 100.0%
Total tokens: 12520
Tokens per classified ticket: 85.17

ZERO-SHOT CONFUSION TABLE
Predicted  billing  delivery  fraud  technical
Actual                                        
billing         38         0      0          0
delivery         0        36      0          0
fraud           11         0     23          0
technical       10         0      0         29


In [24]:
from src.triage import run_pipeline, ZERO

budget_result, budget_stats = run_pipeline(
    prompt=ZERO,
    token_budget=6000
)

print("Budget demo status counts:")
print(budget_result["status"].value_counts())

print("\nTotal tokens used:", budget_stats["tokens"])

print("\nBudget exhausted tickets:")
print(
    budget_result[
        budget_result["status"] == "budget_exhausted"
    ][["ticket_id", "status", "route"]]
)

print("\nOutput file:")
print("outputs/triage_budget_demo.csv")

retry ticket=T0002 retry=0 wait=0.0530s
retry ticket=T0116 retry=0 wait=0.0453s
retry ticket=T0153 retry=0 wait=0.0484s
retry ticket=T0039 retry=0 wait=0.0478s
retry ticket=T0152 retry=0 wait=0.0542s
retry ticket=T0043 retry=0 wait=0.0535s
retry ticket=T0111 retry=0 wait=0.0562s
retry ticket=T0155 retry=0 wait=0.0461s
retry ticket=T0100 retry=0 wait=0.0503s
retry ticket=T0068 retry=0 wait=0.0454s
retry ticket=T0135 retry=0 wait=0.0477s
retry ticket=T0009 retry=0 wait=0.0513s
retry ticket=T0067 retry=0 wait=0.0453s
retry ticket=T0106 retry=0 wait=0.0475s
retry ticket=T0097 retry=0 wait=0.0531s
Budget demo status counts:
status
classified_json           111
budget_exhausted           29
classified_keyword          9
flagged_invalid_output      7
skipped_empty               4
flagged_long                4
Name: count, dtype: int64

Total tokens used: 6009

Budget exhausted tickets:
    ticket_id            status         route
132     T0032  budget_exhausted  human_review
133     T0003  b

In [25]:
from src.triage import run_pipeline, FEW

final_result, final_stats = run_pipeline(
    prompt=FEW,
    token_budget=100000
)

print("Final status counts:")
print(final_result["status"].value_counts())

print("\nTotal tokens:", final_stats["tokens"])

print("\nRouting counts:")
print(final_result["route"].value_counts())

print("\nPriority counts:")
print(final_result["priority"].value_counts())

print("\nOutput shape:", final_result.shape)
print("\nOutput columns:")
print(final_result.columns.tolist())

retry ticket=T0002 retry=0 wait=0.0530s
retry ticket=T0116 retry=0 wait=0.0453s
retry ticket=T0153 retry=0 wait=0.0484s
retry ticket=T0039 retry=0 wait=0.0478s
retry ticket=T0152 retry=0 wait=0.0542s
retry ticket=T0043 retry=0 wait=0.0535s
retry ticket=T0111 retry=0 wait=0.0562s
retry ticket=T0155 retry=0 wait=0.0461s
retry ticket=T0100 retry=0 wait=0.0503s
retry ticket=T0068 retry=0 wait=0.0454s
retry ticket=T0135 retry=0 wait=0.0477s
retry ticket=T0009 retry=0 wait=0.0513s
retry ticket=T0067 retry=0 wait=0.0453s
retry ticket=T0106 retry=0 wait=0.0475s
retry ticket=T0097 retry=0 wait=0.0531s
retry ticket=T0003 retry=0 wait=0.0518s
retry ticket=T0019 retry=0 wait=0.0478s
retry ticket=T0086 retry=0 wait=0.0524s
retry ticket=T0065 retry=0 wait=0.0551s
Final status counts:
status
classified_json           135
classified_keyword         12
flagged_invalid_output      9
skipped_empty               4
flagged_long                4
Name: count, dtype: int64

Total tokens: 12520

Routing counts

In [26]:
import pandas as pd

final_csv = pd.read_csv("outputs/triage_results.csv", keep_default_na=False)
segments = pd.read_csv("outputs/customer_segments.csv")

print("CSV shape:", final_csv.shape)
print("Duplicate ticket IDs:", final_csv["ticket_id"].duplicated().sum())
print("Missing values:")
print(final_csv.isna().sum())

print("\nHuman review routing:")
print(
    final_csv[
        final_csv["status"].isin([
            "skipped_empty",
            "flagged_long",
            "flagged_invalid_output",
            "budget_exhausted"
        ])
    ]["route"].value_counts()
)

print("\nFraud routing:")
print(
    final_csv[final_csv["category"] == "fraud"]["route"].value_counts()
)

print("\nFinal CSV exists and has 164 rows:", len(final_csv) == 164)

FileNotFoundError: [Errno 2] No such file or directory: 'outputs/triage_results.csv'

In [27]:
from src.triage import OUT

print("Output folder:", OUT)
print("\nFiles available:")

for file in OUT.iterdir():
    print(file.name)

Output folder: C:\instamart_customer_copilot_teaching_kit\outputs

Files available:
customer_segments.csv
lapse_model.joblib
retention_list.csv
triage_budget_demo.csv
triage_results.csv


In [28]:
import pandas as pd

final_csv = pd.read_csv(OUT / "triage_results.csv", keep_default_na=False)
segments = pd.read_csv(OUT / "customer_segments.csv")

print("CSV shape:", final_csv.shape)
print("Duplicate ticket IDs:", final_csv["ticket_id"].duplicated().sum())

print("\nMissing values:")
print(final_csv.isna().sum())

print("\nHuman review routing:")
print(final_csv[
    final_csv["status"].isin([
        "skipped_empty",
        "flagged_long",
        "flagged_invalid_output",
        "budget_exhausted"
    ])
]["route"].value_counts())

print("\nFraud routing:")
print(final_csv[final_csv["category"] == "fraud"]["route"].value_counts())

print("\nFinal CSV has 164 rows:", len(final_csv) == 164)

CSV shape: (164, 8)
Duplicate ticket IDs: 0

Missing values:
ticket_id       0
customer_id     0
status          0
category        0
segment_name    0
priority        0
route           0
tokens_used     0
dtype: int64

Human review routing:
route
human_review    17
Name: count, dtype: int64

Fraud routing:
route
security_team    34
Name: count, dtype: int64

Final CSV has 164 rows: True


In [29]:
import pandas as pd
import re
from src.triage import OUT, RAW, redact, PATTERNS

# Load files
tickets = pd.read_csv(RAW / "tickets.csv", keep_default_na=False)
final_df = pd.read_csv(OUT / "triage_results.csv", keep_default_na=False)
budget_df = pd.read_csv(OUT / "triage_budget_demo.csv", keep_default_na=False)
segments = pd.read_csv(OUT / "customer_segments.csv", keep_default_na=False)

# 1. Privacy audit across ALL 164 tickets
redacted_texts = [redact(str(t))[0] for t in tickets["text"]]

remaining_pii = sum(
    len(pattern.findall(text))
    for text in redacted_texts
    for _, pattern, _ in PATTERNS
)

print("PRIVACY AUDIT")
print("Tickets:", len(redacted_texts))
print("Remaining PII matches:", remaining_pii)

assert remaining_pii == 0

# 2. Empty and long tickets must not trigger model calls
guarded = final_df[
    final_df["status"].isin(["skipped_empty", "flagged_long"])
]

print("\nINPUT GUARDRAIL AUDIT")
print("Guarded tickets:", len(guarded))
print("Tokens used by guarded tickets:", guarded["tokens_used"].sum())

assert (guarded["tokens_used"] == 0).all()

# 3. Verify every routing decision
segment_map = segments.set_index("customer_id").to_dict("index")

def expected_route(row):
    status = row["status"]
    category = row["category"]
    customer = segment_map.get(row["customer_id"], {})

    premium_value = customer.get("is_premium", False)
    is_premium = str(premium_value).strip().lower() == "true"

    if status in {
        "skipped_empty",
        "flagged_long",
        "flagged_invalid_output",
        "budget_exhausted"
    }:
        return "human_review", "NORMAL"

    if category == "fraud":
        return "security_team", "HIGH"

    if is_premium:
        return "premium_desk", "HIGH"

    return "support_queue", "NORMAL"

expected = final_df.apply(expected_route, axis=1)

routing_correct = all(
    (final_df.iloc[i]["route"], final_df.iloc[i]["priority"]) == expected.iloc[i]
    for i in range(len(final_df))
)

print("\nROUTING AUDIT")
print("All routing decisions correct:", routing_correct)

assert routing_correct

# 4. Output file checks
print("\nOUTPUT AUDIT")
print("Final CSV shape:", final_df.shape)
print("Budget CSV shape:", budget_df.shape)
print("Duplicate ticket IDs:", final_df["ticket_id"].duplicated().sum())

assert final_df.shape == (164, 8)
assert budget_df.shape[0] == 164
assert final_df["ticket_id"].is_unique

print("\nALL FINAL PART C CHECKS PASSED")

PRIVACY AUDIT
Tickets: 164
Remaining PII matches: 0

INPUT GUARDRAIL AUDIT
Guarded tickets: 8
Tokens used by guarded tickets: 0

ROUTING AUDIT
All routing decisions correct: True

OUTPUT AUDIT
Final CSV shape: (164, 8)
Budget CSV shape: (164, 8)
Duplicate ticket IDs: 0

ALL FINAL PART C CHECKS PASSED


## PART D - Task 1 


In [30]:
from pathlib import Path

ROOT = Path.cwd()
POLICY_DIR = ROOT / "data" / "policies"

print("Project directory:", ROOT)
print("Policy directory exists:", POLICY_DIR.exists())

for file in sorted(POLICY_DIR.glob("*.txt")):
    word_count = len(file.read_text(encoding="utf-8").split())
    print(f"{file.name}: {word_count} words")

Project directory: c:\instamart_customer_copilot_teaching_kit\notebooks
Policy directory exists: True
delivery_policy.txt: 100 words
escalation_policy.txt: 78 words
payment_policy.txt: 86 words
privacy_policy.txt: 68 words
refund_policy.txt: 119 words


In [31]:
policy_chunks = []

CHUNK_SIZE = 40
OVERLAP = 10
STEP = CHUNK_SIZE - OVERLAP

for file in sorted(POLICY_DIR.glob("*.txt")):
    words = file.read_text(encoding="utf-8").split()

    for start in range(0, len(words), STEP):
        chunk_words = words[start:start + CHUNK_SIZE]

        policy_chunks.append({
            "source": file.name,
            "text": " ".join(chunk_words)
        })

        if start + CHUNK_SIZE >= len(words):
            break

print("Total chunks:", len(policy_chunks))

print("\nChunks per document:")
for file in sorted(POLICY_DIR.glob("*.txt")):
    count = sum(
        chunk["source"] == file.name
        for chunk in policy_chunks
    )
    print(file.name, ":", count)

print("\nFirst chunk:")
print(policy_chunks[0])

Total chunks: 15

Chunks per document:
delivery_policy.txt : 3
escalation_policy.txt : 3
payment_policy.txt : 3
privacy_policy.txt : 2
refund_policy.txt : 4

First chunk:
{'source': 'delivery_policy.txt', 'text': 'Delivery Policy. Standard delivery is promised within 30 minutes of order confirmation in all supported cities. The supported cities are Chennai, Mumbai, Bengaluru, Delhi, Hyderabad and Pune. A delivery fee of INR 25 is charged on orders below INR 199,'}


## 7. Part D — policy RAG

The retrieval layer chunks policy documents, fits a TF-IDF vectorizer and ranks chunks by cosine similarity. It passes retrieved chunks to the offline policy assistant, which should answer only from the context and fall back when a question is not covered.

In [18]:
from src.policy_rag import PolicyRAG, evaluate
rag = PolicyRAG(chunk_size=40, overlap=10)
print("Policy chunks:", len(rag.chunks))
display(pd.DataFrame(rag.chunks).head())
print(rag.answer("What delivery time makes a completed order eligible for a full refund?"))
policy_eval = evaluate(rag)
display(policy_eval)
print("Expected-document hits:", int(policy_eval.retrieved_doc.sum()))
print("Expected-fact hits:", int(policy_eval.fact_found.sum()))
print("Fallbacks:", int(policy_eval.fallback.sum()))

Policy chunks: 15


,source,text
0,delivery_policy.txt,Delivery Policy. Standard delivery is promised...
1,delivery_policy.txt,"of INR 25 is charged on orders below INR 199, ..."
2,delivery_policy.txt,60 minutes and the app shows a rain notice. A ...
3,escalation_policy.txt,Escalation Policy. A ticket about fraud or an ...
4,escalation_policy.txt,"1 hour, and all other tickets within 24 hours...."


{'answer': 'order is delivered in more than 45 minutes, the customer receives a full refund of the order amount.', 'sources': ['refund_policy.txt', 'privacy_policy.txt'], 'retrieved': [{'source': 'refund_policy.txt', 'score': 0.35464416504570484, 'text': 'order is delivered in more than 45 minutes, the customer receives a full refund of the order amount. If a completed order is delivered in 45 minutes or less, no late-delivery refund is paid. Refunds up to INR 1000 are'}, {'source': 'refund_policy.txt', 'score': 0.31425507733397284, 'text': 'Refund Policy. A refund can be requested only for orders whose status is completed. Orders with status cancelled were never charged, so no refund applies to them. If a completed order is delivered in more than 45 minutes, the customer'}, {'source': 'privacy_policy.txt', 'score': 0.15759666156595936, 'text': 'Delivery addresses are deleted from active systems 180 days after the last order. A customer can ask for account deletion and the request is c

,question,expected_doc,retrieved_doc,fact_found,fallback
0,What delivery time makes a completed order eli...,refund_policy.txt,True,True,False
1,Who must approve a refund above INR 1000?,refund_policy.txt,True,True,False
2,How many days do I have to raise a refund requ...,refund_policy.txt,True,True,False
3,What is the delivery fee on small orders?,delivery_policy.txt,True,True,False
4,On orders of what value is delivery free?,delivery_policy.txt,True,True,False
5,Why does the electronics delivery need a one t...,delivery_policy.txt,True,True,False
6,Is cash on delivery available for a large order?,payment_policy.txt,True,True,False
7,How soon is money returned when a payment fail...,payment_policy.txt,True,True,False
8,Can wallet balance be refunded as cash?,payment_policy.txt,True,True,False
9,When are delivery addresses deleted from activ...,privacy_policy.txt,True,True,False


Expected-document hits: 12
Expected-fact hits: 12
Fallbacks: 2


In [32]:
# Repeat evaluation with zero overlap
rag_zero = PolicyRAG(chunk_size=40, overlap=0)

print("Zero-overlap chunks:", len(rag_zero.chunks))

zero_eval = evaluate(rag_zero)

display(zero_eval)

print("\nZERO-OVERLAP RESULTS")
print("Expected-document hits:", int(zero_eval["retrieved_doc"].sum()))
print("Expected-fact hits:", int(zero_eval["fact_found"].sum()))
print("Fallbacks:", int(zero_eval["fallback"].sum()))

print("\nQuestions that failed:")
display(
    zero_eval[
        (~zero_eval["retrieved_doc"]) |
        (~zero_eval["fact_found"])
    ][["question", "expected_doc", "retrieved_doc", "fact_found"]]
)

Zero-overlap chunks: 13


TypeError: evaluate() missing 2 required positional arguments: 'name' and 'total_tokens'

In [33]:
import inspect

print("Function signature:", inspect.signature(evaluate))
print("\nFunction source:")
print(inspect.getsource(evaluate))

Function signature: (result, name, total_tokens)

Function source:
def evaluate(result, name, total_tokens):
    comparison = result.merge(
        tickets[["ticket_id", "true_category"]],
        on="ticket_id",
        how="left"
    )

    classified = comparison[
        comparison["status"].isin(
            ["classified_json", "classified_keyword"]
        )
    ]

    accuracy = (
        classified["category"] == classified["true_category"]
    ).mean() * 100

    print(f"\n{name}")
    print("Classified tickets:", len(classified))
    print(f"Accuracy: {accuracy:.1f}%")
    print("Total tokens:", total_tokens)
    print(
        "Tokens per classified ticket:",
        round(total_tokens / len(classified), 2)
    )

    return comparison, classified



In [34]:
from src.policy_rag import evaluate as evaluate_policy

rag_zero = PolicyRAG(chunk_size=40, overlap=0)

print("Zero-overlap chunks:", len(rag_zero.chunks))

zero_eval = evaluate_policy(rag_zero)

display(zero_eval)

print("\nZERO-OVERLAP RESULTS")
print("Expected-document hits:", int(zero_eval["retrieved_doc"].sum()))
print("Expected-fact hits:", int(zero_eval["fact_found"].sum()))
print("Fallbacks:", int(zero_eval["fallback"].sum()))

print("\nQuestions that failed:")
display(
    zero_eval[
        (~zero_eval["retrieved_doc"]) |
        (~zero_eval["fact_found"])
    ][["question", "expected_doc", "retrieved_doc", "fact_found"]]
)

Zero-overlap chunks: 13


,question,expected_doc,retrieved_doc,fact_found,fallback
0,What delivery time makes a completed order eli...,refund_policy.txt,True,False,False
1,Who must approve a refund above INR 1000?,refund_policy.txt,True,True,False
2,How many days do I have to raise a refund requ...,refund_policy.txt,True,True,False
3,What is the delivery fee on small orders?,delivery_policy.txt,True,True,False
4,On orders of what value is delivery free?,delivery_policy.txt,True,True,False
5,Why does the electronics delivery need a one t...,delivery_policy.txt,True,True,False
6,Is cash on delivery available for a large order?,payment_policy.txt,True,True,False
7,How soon is money returned when a payment fail...,payment_policy.txt,True,False,False
8,Can wallet balance be refunded as cash?,payment_policy.txt,True,True,False
9,When are delivery addresses deleted from activ...,privacy_policy.txt,True,True,False



ZERO-OVERLAP RESULTS
Expected-document hits: 12
Expected-fact hits: 10
Fallbacks: 2

Questions that failed:


,question,expected_doc,retrieved_doc,fact_found
0,What delivery time makes a completed order eli...,refund_policy.txt,True,False
7,How soon is money returned when a payment fail...,payment_policy.txt,True,False
12,What is the favourite colour of the company ma...,NONE,False,False
13,Who won the cricket world cup final?,NONE,False,False


In [35]:
eval_data = pd.read_csv(
    ROOT / "data" / "raw" / "policy_eval.csv",
    keep_default_na=False
)

failed = zero_eval[
    (zero_eval["expected_doc"] != "NONE") &
    (~zero_eval["fact_found"])
]

for _, row in failed.iterrows():
    original = eval_data[
        eval_data["question"] == row["question"]
    ].iloc[0]

    print("\nQuestion:", original["question"])
    print("Expected fact:", original["expected_fact"])
    print("Policy:", original["expected_doc"])

    print("\nZero-overlap chunks from that policy:")
    for chunk in rag_zero.chunks:
        if chunk["source"] == original["expected_doc"]:
            print("-", chunk["text"])


Question: What delivery time makes a completed order eligible for a full refund?
Expected fact: more than 45 minutes
Policy: refund_policy.txt

Zero-overlap chunks from that policy:
- Refund Policy. A refund can be requested only for orders whose status is completed. Orders with status cancelled were never charged, so no refund applies to them. If a completed order is delivered in more than 45 minutes, the customer
- receives a full refund of the order amount. If a completed order is delivered in 45 minutes or less, no late-delivery refund is paid. Refunds up to INR 1000 are approved automatically by the support system. A refund above INR
- 1000 must be approved by a human support lead before money is returned. Approved refunds are credited to the original payment method within 5 working days. A refund request must be raised within 7 days of the delivery date.

Question: How soon is money returned when a payment fails but is deducted?
Expected fact: 48 hours
Policy: payment_policy.txt

In [38]:
print("Tool count:", len(TOOLS_SPEC))
print("Tool map count:", len(TOOL_MAP))

for tool in TOOLS_SPEC:
    print("\nTool:", tool["name"])
    print("Description:", tool["description"])
    print("Parameters:", list(tool["parameters"]["properties"]))

assert len(TOOLS_SPEC) == 5
assert set(t["name"] for t in TOOLS_SPEC) == set(TOOL_MAP)

print("\nALL FIVE TOOLS VERIFIED")

NameError: name 'TOOLS_SPEC' is not defined

In [39]:
import importlib
import src.refund_agent as refund_agent

# Reload the updated Python file
importlib.reload(refund_agent)

# Import both variables into the notebook
TOOLS_SPEC = refund_agent.TOOLS_SPEC
TOOL_MAP = refund_agent.TOOL_MAP

print("Tool count:", len(TOOLS_SPEC))
print("Tool map count:", len(TOOL_MAP))

for tool in TOOLS_SPEC:
    print("\nTool:", tool["name"])
    print("Description:", tool["description"])
    print("Parameters:", list(tool["parameters"]["properties"]))

assert len(TOOLS_SPEC) == 5
assert set(t["name"] for t in TOOLS_SPEC) == set(TOOL_MAP)

print("\nALL FIVE TOOLS VERIFIED")

Tool count: 5
Tool map count: 5

Tool: get_order
Description: Look up an order in the Instamart database using its order ID.
Parameters: ['order_id']

Tool: search_policy
Description: Search the policy documents and return the most relevant policy chunks.
Parameters: ['query']

Tool: calculate_refund
Description: Calculate the eligible refund based on order amount and delivery time.
Parameters: ['order_amount', 'delivery_minutes']

Tool: issue_refund
Description: Record an approved automatic refund in the refund ledger.
Parameters: ['order_id', 'refund_inr']

Tool: escalate_to_human
Description: Record a refund requiring human approval in the refund ledger.
Parameters: ['order_id', 'refund_inr']

ALL FIVE TOOLS VERIFIED


In [44]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

print("Searching for agent_scenarios.csv...\n")

matches = list(PROJECT_ROOT.rglob("agent_scenarios.csv"))

if matches:
    for file in matches:
        print("FOUND:", file)
else:
    print("agent_scenarios.csv was not found.")

print("\nFiles inside data/raw:")
raw_dir = PROJECT_ROOT / "data" / "raw"

if raw_dir.exists():
    for file in sorted(raw_dir.iterdir()):
        print(file.name)
else:
    print("data/raw folder does not exist.")

Searching for agent_scenarios.csv...

agent_scenarios.csv was not found.

Files inside data/raw:
customers.csv
orders.csv
policy_eval.csv
products.csv
tickets.csv


In [45]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

for item in sorted(PROJECT_ROOT.iterdir()):
    print(item.name)

.gitignore
.venv
.venv-1
__pycache__
data
generate_data.py
GITHUB_RESOURCES.md
mock_llm.py
notebooks
outputs
README.md
requirements.txt
run_project.py
sql
src
state


In [46]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

scenario_file = PROJECT_ROOT / "data" / "raw" / "agent_scenarios.csv"

print("File exists:", scenario_file.exists())

scenarios = pd.read_csv(scenario_file, keep_default_na=False)

display(scenarios)
print("Total scenarios:", len(scenarios))

File exists: True


,scenario_id,customer_id,message,expected_outcome,expected_refund_inr
0,S1,C0007,My order ORD-100048 reached me very late. I wo...,REFUND,128.25
1,S2,C0008,My order ORD-100067 reached me very late. I wo...,REFUND,135.00
2,S3,C0016,"Order ORD-100145 arrived far too late, please ...",ESCALATE,1999.00
3,S4,C0001,"Please refund order ORD-100001, I did not like...",DENIED,0.00
4,S5,C0003,I want my money back for order ORD-100025.,DENIED,0.00
5,S6,C0001,Refund for order ORD-999999 please.,NOT_FOUND,0.00
6,S7,C0007,[LOOP] Refund order ORD-100048.,FALLBACK,0.00


Total scenarios: 7


In [18]:
from pathlib import Path
import pandas as pd

for name in ["triage_results.csv", "triage_budget_demo.csv"]:
    path = Path("outputs") / name
    print(f"\n{name}")
    print("Exists:", path.exists())

    if path.exists():
        df = pd.read_csv(path)
        print("Rows:", len(df))
        print(df["status"].value_counts())


triage_results.csv
Exists: True
Rows: 164
status
classified_json           135
classified_keyword         12
flagged_invalid_output      9
skipped_empty               4
flagged_long                4
Name: count, dtype: int64

triage_budget_demo.csv
Exists: True
Rows: 164
status
classified_json           111
budget_exhausted           29
classified_keyword          9
flagged_invalid_output      7
skipped_empty               4
flagged_long                4
Name: count, dtype: int64


In [19]:
import pandas as pd

result = pd.read_csv("outputs/triage_results.csv")
tickets = pd.read_csv("data/raw/tickets.csv", keep_default_na=False)

comparison = result.merge(
    tickets[["ticket_id", "true_category"]],
    on="ticket_id",
    how="left"
)

classified = comparison[
    comparison["status"].isin(
        ["classified_json", "classified_keyword"]
    )
]

accuracy = (
    classified["category"] == classified["true_category"]
).mean() * 100

print("Classified tickets:", len(classified))
print(f"Few-shot accuracy: {accuracy:.2f}%")
print("\nRouting distribution:")
print(result["route"].value_counts())
print("\nPriority distribution:")
print(result["priority"].value_counts())

Classified tickets: 147
Few-shot accuracy: 100.00%

Routing distribution:
route
support_queue    102
security_team     34
human_review      17
premium_desk      11
Name: count, dtype: int64

Priority distribution:
priority
NORMAL    119
HIGH       45
Name: count, dtype: int64


In [20]:
import pandas as pd

result = pd.read_csv("outputs/triage_results.csv")
segments = pd.read_csv("outputs/customer_segments.csv")

segment_map = segments.set_index("customer_id")["is_premium"].to_dict()

def expected_route(row):
    if row["status"] in [
        "skipped_empty", "flagged_long",
        "flagged_invalid_output", "budget_exhausted"
    ]:
        return "human_review"

    if row["category"] == "fraud":
        return "security_team"

    if str(segment_map.get(row["customer_id"], False)).lower() == "true":
        return "premium_desk"

    return "support_queue"

result["expected_route"] = result.apply(expected_route, axis=1)

print("Routing mismatches:",
      (result["route"] != result["expected_route"]).sum())

print("Total tickets:", len(result))

Routing mismatches: 0
Total tickets: 164


## 8. Part D — refund tools and safe stopping

The refund agent demonstrates a tool loop: look up an order in SQLite, retrieve policy, calculate the eligible refund, then issue it or escalate to a human. In this teaching kit the ledger is a local JSONL file; no money moves.

Before running agent demonstrations, inspect `src/refund_agent.py` and ask students to trace how a tool name maps to a Python function.

In [19]:
from src.refund_agent import get_order, search_policy, calculate_refund, TOOL_MAP, run_agent
print("Tools:", sorted(TOOL_MAP))
print("Example order lookup:")
with sqlite3.connect(db_path) as con:
    example_order = pd.read_sql_query(
        "SELECT order_id FROM orders WHERE status='completed' AND delivery_minutes > 45 LIMIT 1", con
    ).iloc[0, 0]
print(example_order, get_order(example_order))
print("Refund calculation:", calculate_refund(250.0, 52))
print("Timeout safety demo:", run_agent(f"Refund order {example_order}", scenario_id="timeout-demo", time_limit_seconds=0))

Tools: ['calculate_refund', 'escalate_to_human', 'get_order', 'issue_refund', 'search_policy']
Example order lookup:
ORD-100048 {'order_id': 'ORD-100048', 'status': 'completed', 'order_amount': 128.25, 'delivery_minutes': 65}
Refund calculation: {'refund_inr': 250.0}
Timeout safety demo: {'status': 'FALLBACK_TIMEOUT', 'final_answer': 'FALLBACK_TIMEOUT: hand off to a human agent.', 'steps': [], 'llm_calls': 0, 'tokens_used': 0}


In [47]:
import importlib
import src.refund_agent as refund_agent

importlib.reload(refund_agent)

agent_results = []

for _, scenario in scenarios.iterrows():
    result = refund_agent.run_agent(
        user_message=scenario["message"],
        scenario_id=scenario["scenario_id"]
    )

    agent_results.append({
        "scenario_id": scenario["scenario_id"],
        "expected_outcome": scenario["expected_outcome"],
        "actual_status": result["status"],
        "expected_refund": scenario["expected_refund_inr"],
        "llm_calls": result["llm_calls"],
        "tokens_used": result["tokens_used"]
    })

agent_results_df = pd.DataFrame(agent_results)
display(agent_results_df)

,scenario_id,expected_outcome,actual_status,expected_refund,llm_calls,tokens_used
0,S1,REFUND,REFUND,128.25,5,897
1,S2,REFUND,REFUND,135.00,5,897
2,S3,ESCALATE,ESCALATE,1999.00,5,884
3,S4,DENIED,DENIED,0.00,4,605
4,S5,DENIED,DENIED,0.00,3,337
5,S6,NOT_FOUND,NOT_FOUND,0.00,2,72
6,S7,FALLBACK,FALLBACK_MAX_ITERATIONS,0.00,6,3000


In [48]:
from src.refund_agent import run_agent

# Token limit demonstration
token_demo = run_agent(
    user_message=scenarios.loc[0, "message"],
    scenario_id="S1-token-demo",
    token_limit=200
)

print("Token demo:", token_demo["status"])
print("Tokens used:", token_demo["tokens_used"])

# Timeout demonstration
timeout_demo = run_agent(
    user_message=scenarios.loc[0, "message"],
    scenario_id="S1-timeout-demo",
    time_limit_seconds=0
)

print("\nTimeout demo:", timeout_demo["status"])
print("Model calls:", timeout_demo["llm_calls"])

Token demo: FALLBACK_TOKEN_LIMIT
Tokens used: 354

Timeout demo: FALLBACK_TIMEOUT
Model calls: 0


In [49]:
from pathlib import Path
import json

STATE = Path.cwd().parent / "state"

print("State directory:", STATE)
print("Checkpoint files:", list(STATE.glob("*.json")))

State directory: c:\instamart_customer_copilot_teaching_kit\state
Checkpoint files: [WindowsPath('c:/instamart_customer_copilot_teaching_kit/state/41024251-dc1d-4f92-adc9-749e9d72499d.json'), WindowsPath('c:/instamart_customer_copilot_teaching_kit/state/57c84e41-941b-4506-ac1b-1edac82732f7.json'), WindowsPath('c:/instamart_customer_copilot_teaching_kit/state/80bf9b54-977b-4f69-8e09-b9d132770959.json'), WindowsPath('c:/instamart_customer_copilot_teaching_kit/state/ad79ab7f-605a-43e4-9441-b89c8638f28b.json'), WindowsPath('c:/instamart_customer_copilot_teaching_kit/state/b4b05ecb-8bc4-45f1-8292-0d2343598aaf.json'), WindowsPath('c:/instamart_customer_copilot_teaching_kit/state/cf7079ce-687a-45d6-a555-6a3945a93364.json'), WindowsPath('c:/instamart_customer_copilot_teaching_kit/state/ff67cb7a-a3d6-46a5-bc3f-5494f92fd0e1.json'), WindowsPath('c:/instamart_customer_copilot_teaching_kit/state/ff7e5a25-7ac4-4d56-b1ff-68a25f6c7428.json')]


In [50]:
import json
from pathlib import Path

STATE = Path.cwd().parent / "state"

for path in sorted(STATE.glob("*.json")):
    data = json.loads(path.read_text(encoding="utf-8"))

    print("\nCheckpoint:", path.name)
    print("Scenario:", data.get("scenario_id"))
    print("Run ID:", data.get("run_id"))
    print("Step number:", data.get("step_number"))
    print("Tools executed:", data.get("tools_executed"))
    print("Tokens used:", data.get("tokens_used"))
    print("LLM calls:", data.get("llm_calls"))


Checkpoint: 41024251-dc1d-4f92-adc9-749e9d72499d.json
Scenario: S2
Run ID: 41024251-dc1d-4f92-adc9-749e9d72499d
Step number: 4
Tools executed: ['get_order', 'search_policy', 'calculate_refund', 'issue_refund']
Tokens used: 621
LLM calls: 4

Checkpoint: 57c84e41-941b-4506-ac1b-1edac82732f7.json
Scenario: S5
Run ID: 57c84e41-941b-4506-ac1b-1edac82732f7
Step number: 2
Tools executed: ['get_order', 'search_policy']
Tokens used: 92
LLM calls: 2

Checkpoint: 80bf9b54-977b-4f69-8e09-b9d132770959.json
Scenario: S7
Run ID: 80bf9b54-977b-4f69-8e09-b9d132770959
Step number: 6
Tools executed: ['search_policy', 'search_policy', 'search_policy', 'search_policy', 'search_policy', 'search_policy']
Tokens used: 3000
LLM calls: 6

Checkpoint: ad79ab7f-605a-43e4-9441-b89c8638f28b.json
Scenario: S4
Run ID: ad79ab7f-605a-43e4-9441-b89c8638f28b
Step number: 3
Tools executed: ['get_order', 'search_policy', 'calculate_refund']
Tokens used: 345
LLM calls: 3

Checkpoint: b4b05ecb-8bc4-45f1-8292-0d2343598aaf.js

In [1]:
import importlib
import src.refund_agent as agent

importlib.reload(agent)

print("resume_agent available:", hasattr(agent, "resume_agent"))

ModuleNotFoundError: No module named 'src'

In [2]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path(r"C:\instamart_customer_copilot_teaching_kit")

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

print("Working directory:", Path.cwd())
print("src exists:", (PROJECT_ROOT / "src").exists())

Working directory: C:\instamart_customer_copilot_teaching_kit
src exists: True


In [3]:
import importlib
import src.refund_agent as agent

importlib.reload(agent)

print("resume_agent available:", hasattr(agent, "resume_agent"))

resume_agent available: True


In [4]:
from src.refund_agent import run_agent, resume_agent
import pandas as pd

scenarios = pd.read_csv(
    "data/raw/agent_scenarios.csv",
    keep_default_na=False
)

s1_message = scenarios.loc[
    scenarios["scenario_id"] == "S1", "message"
].iloc[0]

print("S1 message:", s1_message)

S1 message: My order ORD-100048 reached me very late. I would like a refund.


In [5]:
import json
from pathlib import Path
import mock_llm

run_id = "partD-S1-crash-resume"

# Start with a clean checkpoint for this demonstration
checkpoint_path = agent.STATE / f"{run_id}.json"

if checkpoint_path.exists():
    checkpoint_path.unlink()

mock_llm.reset()

try:
    agent.run_agent(
        user_message=s1_message,
        scenario_id="S1",
        max_iterations=6,
        token_limit=6000,
        time_limit_seconds=10,
        run_id=run_id,
        crash_after=2
    )
except RuntimeError as e:
    print("Simulated crash:", e)

# Inspect the saved checkpoint
checkpoint = json.loads(
    checkpoint_path.read_text(encoding="utf-8")
)

print("\nCheckpoint saved:", checkpoint_path.exists())
print("Tools executed before crash:", checkpoint["tools_executed"])
print("Step number:", checkpoint["step_number"])

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\instamart_customer_copilot_teaching_kit\\state\\partD-S1-crash-resume.json'

In [6]:
import inspect
from pathlib import Path
import json

print("Agent state folder:", agent.STATE)
print("Run ID:", run_id)
print("Checkpoint exists:", checkpoint_path.exists())

print("\nCurrent run_agent signature:")
print(inspect.signature(agent.run_agent))

print("\nCurrent run_agent source:")
print(inspect.getsource(agent.run_agent))

Agent state folder: C:\instamart_customer_copilot_teaching_kit\state
Run ID: partD-S1-crash-resume
Checkpoint exists: False

Current run_agent signature:
(user_message, scenario_id='demo', max_iterations=6, token_limit=6000, time_limit_seconds=10, run_id=None, crash_after=None)

Current run_agent source:
def run_agent(user_message,scenario_id="demo",max_iterations=6,token_limit=6000,time_limit_seconds=10,run_id=None,crash_after=None):
    run_id=run_id or str(uuid.uuid4()); STATE.mkdir(exist_ok=True)
    messages=[{"role":"system","content":"AGENT: Use tools to verify orders and follow refund policy."},{"role":"user","content":user_message}]
    tokens=0; steps=[]; executed=[]; start=time.monotonic(); llm_calls=0



In [8]:
import inspect

source = inspect.getsource(agent.run_agent)

print(source[source.find("def checkpoint"):])

In [9]:
import inspect

source = inspect.getsource(agent.run_agent)

print("Total source characters:", len(source))
print("Checkpoint keyword found:", "checkpoint" in source)

print("\nCOMPLETE RUN_AGENT SOURCE:\n")
print(source)

Total source characters: 417
Checkpoint keyword found: False

COMPLETE RUN_AGENT SOURCE:

def run_agent(user_message,scenario_id="demo",max_iterations=6,token_limit=6000,time_limit_seconds=10,run_id=None,crash_after=None):
    run_id=run_id or str(uuid.uuid4()); STATE.mkdir(exist_ok=True)
    messages=[{"role":"system","content":"AGENT: Use tools to verify orders and follow refund policy."},{"role":"user","content":user_message}]
    tokens=0; steps=[]; executed=[]; start=time.monotonic(); llm_calls=0



In [10]:
from pathlib import Path

agent_file = Path(agent.__file__)

print("File:", agent_file)
print("File exists:", agent_file.exists())

print("\nACTUAL FILE CONTENT:\n")
print(agent_file.read_text(encoding="utf-8"))

File: C:\instamart_customer_copilot_teaching_kit\src\refund_agent.py
File exists: True

ACTUAL FILE CONTENT:

"""Part D: database-backed refund tools and bounded ReAct loop."""
from pathlib import Path
import json,sqlite3,time,uuid
from datetime import datetime,timezone
import mock_llm
from src.policy_rag import PolicyRAG
ROOT=Path(__file__).resolve().parents[1]; DB=ROOT/"data/instamart.db"; OUT=ROOT/"outputs"; STATE=ROOT/"state"
rag=PolicyRAG()
def get_order(order_id):
    with sqlite3.connect(DB) as con:
        row=con.execute("select order_id,status,order_amount,delivery_minutes from orders where order_id=?",(order_id,)).fetchone()
    return {"error":"order not found"} if not row else dict(zip(["order_id","status","order_amount","delivery_minutes"],row))
def search_policy(query): return {"results":rag.retrieve(query,3)}
def calculate_refund(order_amount,delivery_minutes):
    amount=float(order_amount) if delivery_minutes is not None and float(delivery_minutes)>45 else 0.0
    ret

In [11]:
import importlib
import src.refund_agent as agent

importlib.reload(agent)

print("run_agent:", callable(agent.run_agent))
print("resume_agent:", callable(agent.resume_agent))
print("Checkpoint helper:", callable(agent._save_checkpoint))

run_agent: True
resume_agent: True
Checkpoint helper: True


In [12]:
import inspect

print(inspect.signature(agent.run_agent))
print(inspect.signature(agent.resume_agent))
print(inspect.getsource(agent.run_agent))

(user_message, scenario_id='demo', max_iterations=6, token_limit=6000, time_limit_seconds=10, run_id=None, crash_after=None)
(run_id, max_iterations=6, token_limit=6000, time_limit_seconds=10)
def run_agent(
    user_message,
    scenario_id="demo",
    max_iterations=6,
    token_limit=6000,
    time_limit_seconds=10,
    run_id=None,
    crash_after=None
):
    run_id = run_id or str(uuid.uuid4())
    STATE.mkdir(exist_ok=True)

    messages = [
        {"role": "system", "content": "AGENT: Use tools to verify orders and follow refund policy."},
        {"role": "user", "content": user_message}
    ]

    tokens = 0
    steps = []
    executed = []
    start = time.monotonic()
    llm_calls = 0

    for iteration in range(max_iterations):

        if time.monotonic() - start >= time_limit_seconds:
            return {
                "status": "FALLBACK_TIMEOUT",
                "final_answer": "FALLBACK_TIMEOUT: hand off to a human agent.",
                "steps": steps,
          

In [13]:
import uuid
import json

run_id = "partD-S1-crash-resume"

s1 = scenarios.loc[scenarios["scenario_id"] == "S1"].iloc[0]

try:
    agent.run_agent(
        user_message=s1["message"],
        scenario_id="S1",
        run_id=run_id,
        crash_after=2
    )
except RuntimeError as e:
    print("Simulated crash:", e)

checkpoint_path = agent.STATE / f"{run_id}.json"

print("Checkpoint exists:", checkpoint_path.exists())

checkpoint = json.loads(
    checkpoint_path.read_text(encoding="utf-8")
)

print("Tools executed before crash:", checkpoint["tools_executed"])
print("Checkpoint step:", checkpoint["step_number"])

Simulated crash: Simulated crash after checkpoint
Checkpoint exists: True
Tools executed before crash: ['get_order', 'search_policy']
Checkpoint step: 2


In [14]:
resumed = agent.resume_agent(
    run_id="partD-S1-crash-resume"
)

print("Final status:", resumed["status"])
print("Final answer:", resumed["final_answer"])
print("Model calls:", resumed["llm_calls"])
print("Tokens used:", resumed["tokens_used"])

print("\nAll tools executed:")
for step in resumed["steps"]:
    print(step["tool"])

Final status: REFUND
Final answer: REFUND: INR 128.25 issued.
Model calls: 5
Tokens used: 897

All tools executed:
get_order
search_policy
calculate_refund
issue_refund


##### ReAct vs Plan-and-Execute

The ReAct agent makes decisions step by step by alternating between model calls and tool executions. In our experiments, S1 required 5 model calls and 897 tokens, S5 required 3 calls and 337 tokens, and S6 required 2 calls and 72 tokens.

A plan-and-execute agent would ask the model to generate the complete sequence of actions in one planning call and execute those actions using tools. This can reduce the number of model calls and potentially lower latency. However, a complete plan may require more tokens in its initial response, and replanning may be necessary when unexpected situations occur.

ReAct handles unexpected observations more naturally. For example, if an order is not found or its status is cancelled, the agent can inspect the tool result and adjust its next action. A fixed plan-and-execute workflow may continue with an unsuitable action unless error handling and replanning are implemented.

For this refund workflow, I would choose **ReAct** because refund decisions involve checking order status, delivery time, policy conditions and refund limits. Its step-by-step reasoning provides better flexibility and safer handling of unexpected tool results, even though it may require more model calls.

In [15]:
from pathlib import Path
import json

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

required_files = [
    ROOT / "outputs" / "agent_runs.jsonl",
    ROOT / "outputs" / "refund_ledger.jsonl",
]

for file in required_files:
    print(f"\n{file.relative_to(ROOT)}")
    print("Exists:", file.exists())

    if file.exists():
        lines = file.read_text(encoding="utf-8").splitlines()
        print("Records:", len(lines))

        for line in lines[:2]:
            print(json.loads(line))

state_dir = ROOT / "state"
checkpoints = list(state_dir.glob("*.json")) if state_dir.exists() else []

print("\nCheckpoint files:", len(checkpoints))


outputs\agent_runs.jsonl
Exists: False

outputs\refund_ledger.jsonl
Exists: True
Records: 4
{'timestamp': '2026-11-09T08:19:50.410677+00:00', 'tool': 'issue_refund', 'order_id': 'ORD-100048', 'refund_inr': 128.25}
{'timestamp': '2026-11-09T08:19:50.427375+00:00', 'tool': 'issue_refund', 'order_id': 'ORD-100067', 'refund_inr': 135.0}

Checkpoint files: 9


## 9. Final generated artifacts and reproducibility check

Review the CSVs and model file generated by the pipeline. In class, have students delete `data/clean/`, `data/instamart.db`, and `outputs/`, then rerun the documented commands to verify that the project can be rebuilt from source.

**Important:** This teaching kit is an instructional implementation. If the course grades exact numeric checkpoints from the original brief, run every checkpoint and reconcile differences; do not hardcode expected answers.

In [20]:
for folder in [ROOT/"data"/"clean", ROOT/"outputs", ROOT/"state"]:
    print(f"\\n{folder.relative_to(ROOT)}")
    print(sorted(p.name for p in folder.iterdir()) if folder.exists() else "(not created)")
print("Database exists:", db_path.exists())

\ndata\clean
['customers.csv', 'orders.csv', 'products.csv']
\noutputs
['customer_segments.csv', 'lapse_model.joblib', 'retention_list.csv', 'triage_results.csv']
\nstate
[]
Database exists: True


In [16]:
import json
import shutil

agent_runs_path = ROOT / "outputs" / "agent_runs.jsonl"
temp_ledger_dir = ROOT / "outputs" / "_audit_temp"

original_out = agent.OUT
agent.OUT = temp_ledger_dir

run_records = []

try:
    for _, scenario in scenarios.iterrows():
        result = agent.run_agent(
            user_message=scenario["message"],
            scenario_id=scenario["scenario_id"]
        )

        run_records.append({
            "scenario_id": scenario["scenario_id"],
            "status": result["status"],
            "final_answer": result["final_answer"],
            "llm_calls": result["llm_calls"],
            "tokens_used": result["tokens_used"],
            "steps": result["steps"]
        })

finally:
    agent.OUT = original_out
    if temp_ledger_dir.exists():
        shutil.rmtree(temp_ledger_dir)

with agent_runs_path.open("w", encoding="utf-8") as f:
    for record in run_records:
        f.write(json.dumps(record, default=str) + "\n")

print("Agent run log exists:", agent_runs_path.exists())
print("Scenario records:", len(run_records))
print("Statuses:", [r["status"] for r in run_records])

Agent run log exists: True
Scenario records: 7
Statuses: ['REFUND', 'REFUND', 'ESCALATE', 'DENIED', 'DENIED', 'NOT_FOUND', 'FALLBACK_MAX_ITERATIONS']


In [17]:
import json

agent_log = ROOT / "outputs" / "agent_runs.jsonl"
ledger = ROOT / "outputs" / "refund_ledger.jsonl"

runs = [
    json.loads(line)
    for line in agent_log.read_text(encoding="utf-8").splitlines()
]

ledger_rows = [
    json.loads(line)
    for line in ledger.read_text(encoding="utf-8").splitlines()
]

print("Agent run records:", len(runs))
print("Refund ledger records:", len(ledger_rows))

print("\nScenario IDs:", [r["scenario_id"] for r in runs])
print("All runs contain steps:", all("steps" in r for r in runs))
print("All runs contain required fields:", all(
    all(k in r for k in [
        "scenario_id", "status", "final_answer",
        "llm_calls", "tokens_used", "steps"
    ])
    for r in runs
))

print("\nPART D ARTIFACT AUDIT COMPLETE")

Agent run records: 7
Refund ledger records: 4

Scenario IDs: ['S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7']
All runs contain steps: True
All runs contain required fields: True

PART D ARTIFACT AUDIT COMPLETE


In [21]:
from pathlib import Path

ROOT = Path.cwd()

required_files = [
    "README.md",
    "requirements.txt",
    "generate_data.py",
    "mock_llm.py",
    "run_project.py",
    "src/data_pipeline.py",
    "src/customer_models.py",
    "src/triage.py",
    "src/policy_rag.py",
    "src/refund_agent.py",
    "notebooks/Customer_Intelligence_Support_Copilot.ipynb",
    "outputs/triage_results.csv",
    "outputs/triage_budget_demo.csv",
    "outputs/agent_runs.jsonl",
    "outputs/refund_ledger.jsonl",
    "outputs/lapse_model.joblib",
    "outputs/retention_list.csv",
]

for file in required_files:
    path = ROOT / file
    print(("OK      " if path.exists() else "MISSING "), file)

print("\nPROJECT ROOT:", ROOT)

OK       README.md
OK       requirements.txt
OK       generate_data.py
OK       mock_llm.py
OK       run_project.py
OK       src/data_pipeline.py
OK       src/customer_models.py
OK       src/triage.py
OK       src/policy_rag.py
OK       src/refund_agent.py
OK       notebooks/Customer_Intelligence_Support_Copilot.ipynb
OK       outputs/triage_results.csv
OK       outputs/triage_budget_demo.csv
OK       outputs/agent_runs.jsonl
OK       outputs/refund_ledger.jsonl
OK       outputs/lapse_model.joblib
OK       outputs/retention_list.csv

PROJECT ROOT: C:\instamart_customer_copilot_teaching_kit


In [22]:
from pathlib import Path

gitignore = Path(".gitignore")

print("Does .gitignore exist?", gitignore.exists())

if gitignore.exists():
    print("\nCurrent .gitignore:\n")
    print(gitignore.read_text())

Does .gitignore exist? True

Current .gitignore:

.venv/
venv/
__pycache__/
*.py[cod]
.env
data/instamart.db
state/*.json

